# Autoencoders

Basic, denoising, and variational autoencoders — implemented from scratch with full
backpropagation, validated against PyTorch, and trained on sklearn digits (8×8 grayscale).

We answer four concrete questions with numbers:
1. How does an undercomplete autoencoder compress and reconstruct images?
2. Does a nonlinear AE beat linear PCA at the same bottleneck width?
3. Does a denoising autoencoder recover clean digits from corrupted inputs better than a plain AE?
4. What does a VAE's KL penalty buy you — and what does it cost?

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

np.random.seed(0)
plt.rcParams['figure.dpi'] = 100

## 1. From-Scratch Autoencoder, Denoising AE & VAE

An **autoencoder** learns an encoder $f_\theta: \mathbb{R}^D \to \mathbb{R}^K$ and
decoder $g_\phi: \mathbb{R}^K \to \mathbb{R}^D$ to minimize reconstruction error:

$$\mathcal{L} = \frac{1}{ND}\| g_\phi(f_\theta(X)) - X \|^2$$

With $K < D$ (**undercomplete**), the bottleneck forces compression — the network must
discover a compact representation of the data.

A **denoising autoencoder** (Vincent et al., 2008) corrupts inputs $\tilde{X}$ but still
reconstructs the **clean** target $X$, learning features robust to noise.

A **variational autoencoder** (Kingma & Welling, 2014) treats the latent code as a
distribution $q_\theta(z|x) = \mathcal{N}(\mu, \mathrm{diag}(\sigma^2))$ and adds a
KL penalty toward a standard normal prior:

$$\mathcal{L} = \| \hat{X} - X \|^2 + \beta \cdot D_{KL}\!\big(q(z|x)\,\|\,\mathcal{N}(0,I)\big)$$

The **reparameterization trick** $z = \mu + \sigma \odot \epsilon$, $\epsilon \sim \mathcal{N}(0,I)$,
makes the sampling step differentiable.

In [2]:
def relu(x):
    return np.maximum(0, x)


def relu_grad(a):
    return (a > 0).astype(a.dtype)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))


def sigmoid_grad(y):
    return y * (1.0 - y)


class Autoencoder:
    """MLP autoencoder: encoder ReLU→linear bottleneck, decoder ReLU→sigmoid output."""

    def __init__(self, input_dim, hidden, latent, seed=0):
        rng = np.random.RandomState(seed)
        s_in = np.sqrt(2.0 / input_dim)
        s_h = np.sqrt(2.0 / hidden)
        s_z = np.sqrt(2.0 / latent)
        self.We1 = rng.randn(input_dim, hidden) * s_in
        self.be1 = np.zeros(hidden)
        self.We2 = rng.randn(hidden, latent) * s_h
        self.be2 = np.zeros(latent)
        self.Wd1 = rng.randn(latent, hidden) * s_z
        self.bd1 = np.zeros(hidden)
        self.Wd2 = rng.randn(hidden, input_dim) * s_h
        self.bd2 = np.zeros(input_dim)
        self.cache = {}

    def forward(self, X):
        h1 = relu(X @ self.We1 + self.be1)
        z = h1 @ self.We2 + self.be2
        h2 = relu(z @ self.Wd1 + self.bd1)
        out = sigmoid(h2 @ self.Wd2 + self.bd2)
        self.cache = dict(X=X, h1=h1, z=z, h2=h2, out=out)
        return out, z

    def backward(self, d_out):
        X, h1, z, h2, out = self.cache.values()
        n = X.shape[0]
        dh2 = d_out * sigmoid_grad(out)
        dWd2 = h2.T @ dh2 / n
        dbd2 = dh2.mean(0)
        dh2_in = dh2 @ self.Wd2.T
        dz_dec = dh2_in * relu_grad(h2)
        dWd1 = z.T @ dz_dec / n
        dbd1 = dz_dec.mean(0)
        dz = dz_dec @ self.Wd1.T
        dWe2 = h1.T @ dz / n
        dbe2 = dz.mean(0)
        dh1 = dz @ self.We2.T
        dh1_in = dh1 * relu_grad(h1)
        dWe1 = X.T @ dh1_in / n
        dbe1 = dh1_in.mean(0)
        dX = dh1_in @ self.We1.T
        return dict(We1=dWe1, be1=dbe1, We2=dWe2, be2=dbe2,
                    Wd1=dWd1, bd1=dbd1, Wd2=dWd2, bd2=dbd2, dX=dX)

    def param_dict(self):
        return dict(We1=self.We1, be1=self.be1, We2=self.We2, be2=self.be2,
                    Wd1=self.Wd1, bd1=self.bd1, Wd2=self.Wd2, bd2=self.bd2)


class DenoisingAE(Autoencoder):
    def __init__(self, input_dim, hidden, latent, noise_std=0.4, seed=0):
        super().__init__(input_dim, hidden, latent, seed=seed)
        self.noise_std = noise_std

    def corrupt(self, X, rng):
        return np.clip(X + rng.randn(*X.shape) * self.noise_std, 0.0, 1.0)


class VAE:
    def __init__(self, input_dim, hidden, latent, seed=0):
        rng = np.random.RandomState(seed)
        s_in = np.sqrt(2.0 / input_dim)
        s_h = np.sqrt(2.0 / hidden)
        s_z = np.sqrt(2.0 / latent)
        self.We1 = rng.randn(input_dim, hidden) * s_in
        self.be1 = np.zeros(hidden)
        self.We_mu = rng.randn(hidden, latent) * s_h
        self.be_mu = np.zeros(latent)
        self.We_lv = rng.randn(hidden, latent) * s_h
        self.be_lv = np.zeros(latent)
        self.Wd1 = rng.randn(latent, hidden) * s_z
        self.bd1 = np.zeros(hidden)
        self.Wd2 = rng.randn(hidden, input_dim) * s_h
        self.bd2 = np.zeros(input_dim)
        self.cache = {}

    def encode(self, X):
        h = relu(X @ self.We1 + self.be1)
        mu = h @ self.We_mu + self.be_mu
        logvar = h @ self.We_lv + self.be_lv
        return h, mu, logvar

    def reparam(self, mu, logvar, rng):
        eps = rng.randn(*mu.shape)
        return mu + np.exp(0.5 * logvar) * eps, eps

    def decode(self, z):
        h = relu(z @ self.Wd1 + self.bd1)
        return sigmoid(h @ self.Wd2 + self.bd2)

    def forward(self, X, rng):
        h, mu, logvar = self.encode(X)
        z, eps = self.reparam(mu, logvar, rng)
        out = self.decode(z)
        h_dec = relu(z @ self.Wd1 + self.bd1)
        self.cache = dict(X=X, h=h, mu=mu, logvar=logvar, z=z, eps=eps, out=out, h_dec=h_dec)
        return out, mu, logvar

    def loss_parts(self, beta=1.0):
        X = self.cache['X']
        out, mu, logvar = self.cache['out'], self.cache['mu'], self.cache['logvar']
        recon = np.mean((out - X) ** 2)
        kl = -0.5 * np.mean(np.sum(1 + logvar - mu ** 2 - np.exp(logvar), axis=1))
        return recon + beta * kl, recon, kl

    def backward(self, beta=1.0):
        X, h, mu, logvar, z, eps, out, h_dec = self.cache.values()
        n, d = X.shape
        d_out = 2.0 * (out - X) / (n * d)
        dh2 = d_out * sigmoid_grad(out)
        dWd2 = h_dec.T @ dh2 / n
        dbd2 = dh2.mean(0)
        dh2_in = dh2 @ self.Wd2.T
        dz = dh2_in * relu_grad(h_dec)
        dWd1 = z.T @ dz / n
        dbd1 = dz.mean(0)
        dz_in = dz @ self.Wd1.T
        dmu = dz_in + beta * mu / n
        dlv = dz_in * eps * 0.5 * np.exp(0.5 * logvar) + beta * 0.5 * (np.exp(logvar) - 1) / n
        dWe_mu = h.T @ dmu / n
        dbe_mu = dmu.mean(0)
        dWe_lv = h.T @ dlv / n
        dbe_lv = dlv.mean(0)
        dh = dmu @ self.We_mu.T + dlv @ self.We_lv.T
        dh_in = dh * relu_grad(h)
        dWe1 = X.T @ dh_in / n
        dbe1 = dh_in.mean(0)
        dX = dh_in @ self.We1.T
        return dict(We1=dWe1, be1=dbe1, We_mu=dWe_mu, be_mu=dbe_mu,
                    We_lv=dWe_lv, be_lv=dbe_lv, Wd1=dWd1, bd1=dbd1,
                    Wd2=dWd2, bd2=dbd2, dX=dX)

    def param_dict(self):
        return dict(We1=self.We1, be1=self.be1, We_mu=self.We_mu, be_mu=self.be_mu,
                    We_lv=self.We_lv, be_lv=self.be_lv, Wd1=self.Wd1, bd1=self.bd1,
                    Wd2=self.Wd2, bd2=self.bd2)


class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1, self.beta2, self.eps = beta1, beta2, eps
        self.m = None
        self.v = None
        self.t = 0

    def step(self, params, grads):
        if self.m is None:
            self.m = {k: np.zeros_like(p) for k, p in params.items()}
            self.v = {k: np.zeros_like(p) for k, p in params.items()}
        self.t += 1
        for k in params:
            self.m[k] = self.beta1 * self.m[k] + (1 - self.beta1) * grads[k]
            self.v[k] = self.beta2 * self.v[k] + (1 - self.beta2) * grads[k] ** 2
            m_hat = self.m[k] / (1 - self.beta1 ** self.t)
            v_hat = self.v[k] / (1 - self.beta2 ** self.t)
            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


def train_autoencoder(model, X_tr, epochs=200, lr=0.002, batch=128, seed=0, beta=1.0):
    opt = Adam(lr=lr)
    rng = np.random.RandomState(seed)
    for _ in range(epochs):
        idx = rng.permutation(len(X_tr))
        for i in range(0, len(X_tr), batch):
            xb = X_tr[idx[i:i + batch]]
            params = model.param_dict()
            if isinstance(model, DenoisingAE):
                xin = model.corrupt(xb, rng)
                out, _ = model.forward(xin)
                n, d = xb.shape
                d_out = 2.0 * (out - xb) / (n * d)
                grads = model.backward(d_out)
            elif isinstance(model, VAE):
                model.forward(xb, rng)
                grads = model.backward(beta)
            else:
                out, _ = model.forward(xb)
                n, d = xb.shape
                d_out = 2.0 * (out - xb) / (n * d)
                grads = model.backward(d_out)
            grads.pop('dX')
            opt.step(params, grads)


def mse(a, b):
    return float(np.mean((a - b) ** 2))


print('Autoencoder, DenoisingAE, VAE, and Adam defined.')

Autoencoder, DenoisingAE, VAE, and Adam defined.


## 2. PyTorch Validation

We copy weights into a matching PyTorch module (float64) and compare forward pass,
input gradients, and weight gradients. Weight gradients match after accounting for
batch-mean vs sum convention ($\times N$).

In [3]:
ae = Autoencoder(64, 8, 4, seed=42)
Xb = np.random.RandomState(0).randn(4, 64) * 0.1 + 0.5
out, z = ae.forward(Xb)
d_out = np.random.RandomState(1).randn(*out.shape)
grads = ae.backward(d_out)
n = Xb.shape[0]


class PT_AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Linear(64, 8)
        self.enc2 = nn.Linear(8, 4)
        self.dec1 = nn.Linear(4, 8)
        self.dec2 = nn.Linear(8, 64)

    def forward(self, x):
        h1 = torch.relu(self.enc1(x))
        z = self.enc2(h1)
        h2 = torch.relu(self.dec1(z))
        return torch.sigmoid(self.dec2(h2)), z


pt = PT_AE().double()
with torch.no_grad():
    pt.enc1.weight.copy_(torch.from_numpy(ae.We1.T))
    pt.enc1.bias.copy_(torch.from_numpy(ae.be1))
    pt.enc2.weight.copy_(torch.from_numpy(ae.We2.T))
    pt.enc2.bias.copy_(torch.from_numpy(ae.be2))
    pt.dec1.weight.copy_(torch.from_numpy(ae.Wd1.T))
    pt.dec1.bias.copy_(torch.from_numpy(ae.bd1))
    pt.dec2.weight.copy_(torch.from_numpy(ae.Wd2.T))
    pt.dec2.bias.copy_(torch.from_numpy(ae.bd2))

Xt = torch.from_numpy(Xb).requires_grad_(True)
out_t, _ = pt(Xt)
(out_t * torch.from_numpy(d_out)).sum().backward()

print('Autoencoder validation:')
print(f'  forward max diff: {np.max(np.abs(out - out_t.detach().numpy())):.2e}')
print(f'  dX max diff:      {np.max(np.abs(grads["dX"] - Xt.grad.numpy())):.2e}')
print(f'  dWe1 max diff:    {np.max(np.abs(grads["We1"] * n - pt.enc1.weight.grad.numpy().T)):.2e}')
print(f'  dWd2 max diff:    {np.max(np.abs(grads["Wd2"] * n - pt.dec2.weight.grad.numpy().T)):.2e}')

# VAE decoder path
vae = VAE(64, 16, 4, seed=7)
h, mu, lv = vae.encode(Xb[:3])
out_det = vae.decode(mu)

class PTDec(nn.Module):
    def __init__(self):
        super().__init__()
        self.dec1 = nn.Linear(4, 16)
        self.dec2 = nn.Linear(16, 64)
    def forward(self, z):
        return torch.sigmoid(self.dec2(torch.relu(self.dec1(z))))

ptd = PTDec().double()
with torch.no_grad():
    ptd.dec1.weight.copy_(torch.from_numpy(vae.Wd1.T))
    ptd.dec1.bias.copy_(torch.from_numpy(vae.bd1))
    ptd.dec2.weight.copy_(torch.from_numpy(vae.Wd2.T))
    ptd.dec2.bias.copy_(torch.from_numpy(vae.bd2))
out_pt = ptd(torch.from_numpy(mu)).detach().numpy()
print('\nVAE decoder validation (z = mu):')
print(f'  decode max diff: {np.max(np.abs(out_det - out_pt)):.2e}')

Autoencoder validation:
  forward max diff: 1.11e-16
  dX max diff:      3.33e-16
  dWe1 max diff:    5.55e-16
  dWd2 max diff:    4.44e-16

VAE decoder validation (z = mu):
  decode max diff: 2.22e-16


## 3. Latent Bottleneck: Nonlinear AE vs Linear PCA

We train undercomplete autoencoders with latent sizes $K \in \{2, 8, 32\}$ on flattened
8×8 digits ($D=64$) and compare test MSE to **PCA**, which is the optimal *linear*
rank-$K$ approximation.

In [4]:
digits = load_digits()
X = digits.images.reshape(-1, 64).astype(np.float64) / 16.0
y = digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

latent_sizes = [2, 8, 32]
rows = []
for k in latent_sizes:
    ae = Autoencoder(64, 128, k, seed=0)
    train_autoencoder(ae, X_tr, epochs=200, lr=0.002, seed=0)
    out, _ = ae.forward(X_te)
    m_ae = mse(out, X_te)
    pca = PCA(k).fit(X_tr)
    m_pca = mse(pca.inverse_transform(pca.transform(X_te)), X_te)
    rows.append((k, m_ae, m_pca))

print(f"{'K':>4} {'AE MSE':>10} {'PCA MSE':>10} {'AE vs PCA':>12}")
for k, m_ae, m_pca in rows:
    pct = (m_pca - m_ae) / m_pca * 100
    print(f"{k:4d} {m_ae:10.5f} {m_pca:10.5f} {pct:+11.1f}%")

fig, ax = plt.subplots(figsize=(6, 4))
ks = [r[0] for r in rows]
ax.plot(ks, [r[1] for r in rows], '-o', label='Nonlinear AE')
ax.plot(ks, [r[2] for r in rows], '-s', label='PCA (linear AE)')
ax.set_xlabel('latent size K')
ax.set_ylabel('test MSE')
ax.set_title('Reconstruction error vs bottleneck width')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ae_latent_sweep.png', dpi=100, bbox_inches='tight')
plt.show()

   K     AE MSE    PCA MSE    AE vs PCA
   2    0.03943    0.05222       +24.5%
   8    0.01309    0.02386       +45.1%
  32    0.00295    0.00262       -12.8%


## 4. Reconstruction Quality vs Bottleneck

Visual comparison: original digits vs reconstructions from $K=2$ and $K=32$ bottlenecks.

In [5]:
ae2 = Autoencoder(64, 128, 2, seed=0)
ae32 = Autoencoder(64, 128, 32, seed=0)
train_autoencoder(ae2, X_tr, epochs=200, lr=0.002, seed=0)
train_autoencoder(ae32, X_tr, epochs=200, lr=0.002, seed=0)

idx = np.arange(8)
orig = X_te[idx]
rec2, _ = ae2.forward(orig)
rec32, _ = ae32.forward(orig)

fig, axes = plt.subplots(3, 8, figsize=(10, 4))
for j in range(8):
    axes[0, j].imshow(orig[j].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
    axes[1, j].imshow(rec2[j].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
    axes[2, j].imshow(rec32[j].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
    for r in range(3):
        axes[r, j].axis('off')
axes[0, 0].set_ylabel('original')
axes[1, 0].set_ylabel('K=2')
axes[2, 0].set_ylabel('K=32')
plt.suptitle('Autoencoder reconstructions (test digits)')
plt.tight_layout()
plt.savefig('ae_recon_grid.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'K=2 test MSE:  {mse(rec2, orig):.5f}')
print(f'K=32 test MSE: {mse(rec32, orig):.5f}')

K=2 test MSE:  0.05015
K=32 test MSE: 0.00326


## 5. Denoising Autoencoder

We add Gaussian noise ($\sigma=0.4$) to inputs at train and test time. The denoising AE
is trained on corrupted inputs but reconstructs **clean** targets. A plain AE trained only
on clean data is evaluated on the same noisy inputs for comparison.

In [6]:
NOISE = 0.4
rng_test = np.random.RandomState(1)
X_te_noisy = np.clip(X_te + rng_test.randn(*X_te.shape) * NOISE, 0.0, 1.0)

ae_clean = Autoencoder(64, 128, 32, seed=0)
dae = DenoisingAE(64, 128, 32, noise_std=NOISE, seed=0)
train_autoencoder(ae_clean, X_tr, epochs=200, lr=0.002, seed=0)
train_autoencoder(dae, X_tr, epochs=200, lr=0.002, seed=0)

out_basic, _ = ae_clean.forward(X_te_noisy)
out_dae, _ = dae.forward(X_te_noisy)
m_input = mse(X_te_noisy, X_te)
m_basic = mse(out_basic, X_te)
m_dae = mse(out_dae, X_te)

print(f'Noisy input MSE (no model):     {m_input:.5f}')
print(f'Basic AE on noisy input:        {m_basic:.5f}')
print(f'Denoising AE on noisy input:    {m_dae:.5f}')
print(f'DAE improvement over basic AE:  {(m_basic - m_dae) / m_basic * 100:.1f}%')

fig, axes = plt.subplots(4, 8, figsize=(10, 5.5))
show = idx
for j, ix in enumerate(show):
    axes[0, j].imshow(X_te[ix].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
    axes[1, j].imshow(X_te_noisy[ix].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
    axes[2, j].imshow(out_basic[ix].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
    axes[3, j].imshow(out_dae[ix].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
    for r in range(4):
        axes[r, j].axis('off')
for r, lab in enumerate(['clean', 'noisy', 'basic AE', 'denoising AE']):
    axes[r, 0].set_ylabel(lab)
plt.suptitle(f'Denoising at noise sigma={NOISE}')
plt.tight_layout()
plt.savefig('dae_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

Noisy input MSE (no model):     0.08518
Basic AE on noisy input:        0.04902
Denoising AE on noisy input:    0.02485
DAE improvement over basic AE:  49.3%


## 6. Variational Autoencoder

We train a VAE with 2D latent space ($\beta=1$) and compare:
- Reconstruction MSE and mean KL divergence
- 2D latent scatter colored by digit class
- kNN digit classification from 2D codes (VAE vs plain AE vs PCA)

The KL term encourages latents to match $\mathcal{N}(0, I)$, which can wash out class
structure — a deliberate trade-off for a smooth, generative latent space.

In [7]:
vae = VAE(64, 256, 2, seed=0)
train_autoencoder(vae, X_tr, epochs=300, lr=0.001, seed=0, beta=1.0)

rng = np.random.RandomState(0)
out_vae, mu_te, lv_te = vae.forward(X_te, rng)
_, mu_tr, _ = vae.forward(X_tr, rng)
recon_vae = mse(out_vae, X_te)
kl_vae = float(-0.5 * np.mean(np.sum(1 + lv_te - mu_te ** 2 - np.exp(lv_te), axis=1)))

ae_lat2 = Autoencoder(64, 128, 2, seed=0)
train_autoencoder(ae_lat2, X_tr, epochs=200, lr=0.002, seed=0)
_, z_tr = ae_lat2.forward(X_tr)
_, z_te = ae_lat2.forward(X_te)

pca2 = PCA(2).fit(X_tr)
zpc_tr = pca2.transform(X_tr)
zpc_te = pca2.transform(X_te)

knn = KNeighborsClassifier(5)
acc_vae = knn.fit(mu_tr, y_tr).score(mu_te, y_te)
acc_ae = knn.fit(z_tr, y_tr).score(z_te, y_te)
acc_pca = knn.fit(zpc_tr, y_tr).score(zpc_te, y_te)

print(f'VAE test recon MSE: {recon_vae:.5f}')
print(f'VAE mean KL:        {kl_vae:.3f}')
print(f'kNN-5 on 2D codes:  VAE={acc_vae:.3f}  AE={acc_ae:.3f}  PCA={acc_pca:.3f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, codes, title in zip(axes, [mu_te, z_te, zpc_te], ['VAE mu', 'AE z', 'PCA']):
    sc = ax.scatter(codes[:, 0], codes[:, 1], c=y_te, cmap='tab10', s=12, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('$z_1$'); ax.set_ylabel('$z_2$')
plt.suptitle('2D latent spaces colored by digit class (test set)')
plt.tight_layout()
plt.savefig('vae_latent_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

VAE test recon MSE: 0.07407
VAE mean KL:        0.001
kNN-5 on 2D codes:  VAE=0.147  AE=0.756  PCA=0.622


## 7. Latent-Space Interpolation

Linear interpolation between two latent codes $\;z(\alpha) = (1-\alpha) z_A + \alpha z_B\;
should produce smooth morphs for a well-trained VAE decoder.

In [8]:
i0, i1 = 0, 5
_, mu0, _ = vae.forward(X_te[i0:i0+1], rng)
_, mu1, _ = vae.forward(X_te[i1:i1+1], rng)
alphas = np.linspace(0, 1, 10)
fig, axes = plt.subplots(2, len(alphas), figsize=(12, 2.8))
for j, a in enumerate(alphas):
    z = (1 - a) * mu0 + a * mu1
    img = vae.decode(z)[0].reshape(8, 8)
    axes[0, j].imshow(img, cmap='gray', vmin=0, vmax=1)
    axes[0, j].axis('off')
    axes[0, j].set_title(f'{a:.1f}', fontsize=8)
axes[1, 0].imshow(X_te[i0].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title(f'digit {y_te[i0]}', fontsize=8)
axes[1, 0].axis('off')
axes[1, -1].imshow(X_te[i1].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
axes[1, -1].set_title(f'digit {y_te[i1]}', fontsize=8)
axes[1, -1].axis('off')
for j in range(1, len(alphas) - 1):
    axes[1, j].axis('off')
plt.suptitle('VAE latent interpolation (top row) between two test digits (bottom corners)')
plt.tight_layout()
plt.savefig('vae_interpolation.png', dpi=100, bbox_inches='tight')
plt.show()